# Session 01 — From Relative Logits to Predictive Surprise

**Deep question:** Why do relative scores, rather than absolute logits, determine a model's belief and the surprise assigned to an observed token?

We will use the cycle **predict → implement → compare with the reference → perturb → interpret → bound the evidence**. Do not rush to run every cell. Stop at each prediction checkpoint and discuss your model of the mechanism first. Clearly labeled reference solutions and explanations appear immediately after each attempt.

This notebook uses only Python's standard library.

## 0. The objects we are studying

At one token position:

- the model emits one **logit** for every vocabulary candidate: `logits.shape == [V]`;
- softmax converts those scores into a probability distribution: `probabilities.shape == [V]`;
- the **label** is one integer ID naming the observed next token: a scalar;
- negative log-likelihood reads the probability at that label.

Later, batching these objects produces logits `[B,T,V]` and labels `[B,T]` before causal alignment.

In [1]:
from math import exp, isclose, log

TOKENS = ["cat", "dog", "slept"]
LOGITS = [2.0, 1.0, -1.0]
LABEL = 1  # the observed next token is 'dog'

list(zip(TOKENS, LOGITS)), TOKENS[LABEL]

([('cat', 2.0), ('dog', 1.0), ('slept', -1.0)], 'dog')

## 1. Prediction checkpoint — normalized competition

Before coding, discuss these questions qualitatively:

1. Which token should receive the greatest probability, and why?
2. If we add `1000` to every logit, should the probabilities change?
3. If only the `dog` logit increases, which other probabilities must change?

Do not calculate the exact values yet. State the mechanism you expect.

<details>
<summary><strong>Reference reasoning — open after making your prediction</strong></summary>

1. `cat` receives the greatest probability because its logit is greatest. Softmax preserves logit ordering.
2. Adding `1000` to every logit changes no probability. In `exp(z_i + c)`, the shared factor `exp(c)` appears in both numerator and denominator and cancels.
3. Raising only the `dog` logit raises `dog`'s probability and lowers both other probabilities. The probabilities must still sum to one, so one candidate gaining probability forces its competitors to lose some.

The important object is therefore the set of relative logit gaps, not the absolute height of any isolated logit.
</details>

## 2. Implement stable softmax

Softmax is `p_i = exp(z_i) / sum_j exp(z_j)`. Subtracting the maximum logit before exponentiation leaves the probabilities unchanged while preventing overflow. Replace only the `...` expressions.

In [ ]:
def stable_softmax(logits):
    maximum = ...                 # largest logit
    weights = ...                 # exp(z_i - maximum) for every z_i
    total = ...                   # sum of the positive weights
    probabilities = ...           # normalize every weight by total
    return probabilities


probabilities = stable_softmax(LOGITS)
list(zip(TOKENS, probabilities))

### Reference solution — stable softmax

Run this only after attempting the scaffold above. It deliberately redefines `stable_softmax` with the canonical implementation.

In [ ]:
def stable_softmax(logits):
    maximum = max(logits)
    weights = [exp(z_i - maximum) for z_i in logits]
    total = sum(weights)
    probabilities = [weight / total for weight in weights]
    return probabilities


probabilities = stable_softmax(LOGITS)
list(zip(TOKENS, probabilities))

**Why this works:** subtracting `maximum` translates every logit by the same amount, so it preserves all pairwise gaps and therefore preserves softmax probabilities. The largest exponent becomes `exp(0) = 1`; every other exponent is at most one, preventing large positive logits from overflowing. Normalization then turns the positive weights into probabilities summing to one.

For these logits the approximate probabilities are `cat: 0.7054`, `dog: 0.2595`, and `slept: 0.0351`. These values describe competition under this complete candidate set; none can be inferred from its logit alone.

In [ ]:
assert len(probabilities) == len(TOKENS)
assert all(0.0 < p < 1.0 for p in probabilities)
assert isclose(sum(probabilities), 1.0, rel_tol=0.0, abs_tol=1e-12)
assert probabilities.index(max(probabilities)) == LOGITS.index(max(LOGITS))
print("The output is a normalized distribution and preserves the score ranking.")

## 3. Perturbation — absolute height versus relative gaps

We now compare the original logits with logits shifted upward by the same constant. This is not merely a numerical trick: it tests what information softmax preserves.

In [ ]:
shifted_logits = [z + 1000.0 for z in LOGITS]
shifted_probabilities = stable_softmax(shifted_logits)

for token, before, after in zip(TOKENS, probabilities, shifted_probabilities):
    print(f"{token:>6}: before={before:.8f} after={after:.8f}")

assert all(isclose(a, b, rel_tol=0.0, abs_tol=1e-12)
           for a, b in zip(probabilities, shifted_probabilities))

In [ ]:
def naive_softmax(logits):
    weights = [exp(z) for z in logits]
    return [weight / sum(weights) for weight in weights]

try:
    naive_softmax(shifted_logits)
except OverflowError as error:
    print("Naive implementation failed:", type(error).__name__)

print("Stable implementation still works:", stable_softmax(shifted_logits))

### Interpretation checkpoint

Explain why adding the same constant changes every raw score but changes no modeled belief. What does this reveal about interpreting one isolated logit?

<details>
<summary><strong>Reference explanation — constant-shift invariance</strong></summary>

For a shared constant `c`, softmax gives `exp(z_i + c) / sum_j exp(z_j + c)`. Factoring out `exp(c)` from the numerator and denominator makes it cancel exactly. The transformed vectors therefore represent the same distribution even though every raw number differs.

A single logit has no absolute probabilistic meaning. A logit becomes meaningful only relative to the competing logits produced at the same position. Stable softmax exploits this invariance computationally: it subtracts the maximum to avoid overflow without altering the model's belief.
</details>

## 4. Labels and negative log-likelihood

The label is not another model prediction. It is evidence from the dataset: the integer ID of the token that actually followed the context. Implement `-log(p_label)`.

In [ ]:
def nll_from_logits(logits, label):
    probabilities = ...           # call stable_softmax
    target_probability = ...      # index with the integer label
    loss = ...                    # negative natural logarithm
    return loss, target_probability


loss, target_probability = nll_from_logits(LOGITS, LABEL)
print("observed token:", TOKENS[LABEL])
print("target probability:", target_probability)
print("target surprise / NLL:", loss)

### Reference solution — target NLL

In [ ]:
def nll_from_logits(logits, label):
    probabilities = stable_softmax(logits)
    target_probability = probabilities[label]
    loss = -log(target_probability)
    return loss, target_probability


loss, target_probability = nll_from_logits(LOGITS, LABEL)
print("observed token:", TOKENS[LABEL])
print("target probability:", target_probability)
print("target surprise / NLL:", loss)
assert isclose(target_probability, 0.25949646034241913, abs_tol=1e-12)
assert isclose(loss, 1.3490122167681864, abs_tol=1e-12)

**Why this works:** the integer label is an index into the model's full probability distribution. It identifies the observed next token without allocating a vocabulary-sized one-hot vector. Negative log turns high target probability into low surprise and very small target probability into large surprise.

The loss cannot use the raw `dog` logit alone because probability depends on the denominator containing every competitor. Equivalently, target NLL is `logsumexp(logits) - target_logit`: both the target score and the complete competition matter.

### Deep question

The highest-scoring token was `cat`, but the observed label is `dog`. Why does the loss need the complete competition among tokens rather than only the raw `dog` logit? Consider what would happen if every logit increased together.

<details>
<summary><strong>Reference answer — why the full competition matters</strong></summary>

Increasing every logit together leaves the distribution and loss unchanged, proving that the raw target score is insufficient. The same `dog` logit could mean high probability when all competitors are lower or tiny probability when another candidate is much higher. Cross-entropy therefore judges the target's normalized share of evidence, not its isolated score.
</details>

## 5. Code the logit gradient `p - q`

For one-hot cross-entropy, the derivative with respect to logit `z_i` is `p_i - q_i`. The target coordinate has `q_i = 1`; every non-target coordinate has `q_i = 0`.

In [ ]:
def logit_gradient(logits, label):
    probabilities = ...
    target = ...                   # a one-hot list with the same length
    gradient = ...                 # elementwise p_i - q_i
    return probabilities, target, gradient


probabilities, target, gradient = logit_gradient(LOGITS, LABEL)
for token, p_i, q_i, grad_i in zip(TOKENS, probabilities, target, gradient):
    print(f"{token:>6}: p={p_i:.6f} q={q_i:.0f} dL/dz={grad_i:+.6f}")

assert isclose(sum(gradient), 0.0, rel_tol=0.0, abs_tol=1e-12)

### Reference solution — the logit gradient `p - q`

In [ ]:
def logit_gradient(logits, label):
    probabilities = stable_softmax(logits)
    target = [1.0 if i == label else 0.0 for i in range(len(logits))]
    gradient = [p_i - q_i for p_i, q_i in zip(probabilities, target)]
    return probabilities, target, gradient


probabilities, target, gradient = logit_gradient(LOGITS, LABEL)
for token, p_i, q_i, grad_i in zip(TOKENS, probabilities, target, gradient):
    print(f"{token:>6}: p={p_i:.6f} q={q_i:.0f} dL/dz={grad_i:+.6f}")

assert gradient[LABEL] < 0.0
assert all(grad_i > 0.0 for i, grad_i in enumerate(gradient) if i != LABEL)
assert isclose(sum(gradient), 0.0, rel_tol=0.0, abs_tol=1e-12)

**Why this works:** for the target `dog`, `q = 1`, so `p - q` is negative. Gradient descent subtracts this negative value and raises the `dog` logit. For every non-target, `q = 0`, so the gradient equals its positive probability; subtracting it lowers that logit. A more strongly believed wrong candidate receives a larger correction.

The gradients sum to zero. This is the differential form of constant-shift invariance: moving all logits equally cannot change the loss, so the loss has zero gradient in that shared-shift direction.

### Interpretation checkpoint

Gradient descent subtracts the gradient. Explain why this raises the target logit and lowers every non-target logit. Then ask a harder question: why does one-hot supervision on one example not force the trained model to assign probability one to that token in every context?

<details>
<summary><strong>Reference answer — one-hot examples and learned distributions</strong></summary>

One example supplies one observed target, but model parameters are shared across many contexts and training examples. Other examples provide different valid continuations, and similar contexts interact through the same parameters. Across representative data, the expected gradient can balance when predicted probabilities match conditional target frequencies.

A sufficiently large model can still memorize finite examples, so this argument does not guarantee calibrated population probabilities. It explains how one-hot sample-level supervision is compatible with distribution learning; generalization and calibration require separate evidence.
</details>

## 6. From average surprise to perplexity

Perplexity exponentiates mean token NLL. It can be interpreted as an effective equal-choice branching factor under a fixed evaluation contract.

In [ ]:
def perplexity(token_losses):
    mean_nll = ...                 # arithmetic mean of valid token losses
    return ...                     # exp(mean_nll)


equal_four_way_loss = -log(0.25)
print(perplexity([equal_four_way_loss] * 5))
assert isclose(perplexity([equal_four_way_loss] * 5), 4.0, abs_tol=1e-12)

### Reference solution — perplexity

In [ ]:
def perplexity(token_losses):
    if not token_losses:
        raise ValueError("perplexity requires at least one valid token loss")
    mean_nll = sum(token_losses) / len(token_losses)
    return exp(mean_nll)


equal_four_way_loss = -log(0.25)
result = perplexity([equal_four_way_loss] * 5)
print(result)
assert isclose(result, 4.0, abs_tol=1e-12)

**Why this works:** token NLL is measured in natural-log units. Averaging gives the typical log surprise per valid target; exponentiation returns that value to probability scale. If every observed token receives probability `1/4`, every loss is `-log(1/4) = log(4)`, so perplexity is `exp(log(4)) = 4`.

This is an effective equal-choice branching factor, not a literal count of plausible words. It is comparable across models only under a compatible tokenizer and identical evaluation contract.

## 7. Evidence boundary — write before concluding

Complete these statements in your own words:

- **Observation:** After adding the same constant to every logit, ...
- **Mechanistic interpretation:** This happens because ...
- **Observation:** For one-hot cross-entropy, the target gradient was ... while non-target gradients were ...
- **What this notebook supports:** ...
- **What it does not prove about a trained language model:** ...

Bring the completed statements and any surprising output back to the guided discussion before Session 2.

<details>
<summary><strong>Model answer — compare only after writing yours</strong></summary>

- **Observation:** Adding the same constant to every logit produced equal probabilities within the declared numerical tolerance.
- **Mechanistic interpretation:** The shared exponential factor cancels during normalization, so only relative logit gaps determine softmax.
- **Observation:** The target logit gradient was negative, while every non-target gradient was positive; all logit gradients summed to approximately zero.
- **What this notebook supports:** For the implemented finite examples, stable softmax normalized scores correctly, target NLL equaled `-log(p_target)`, the analytical one-hot gradient had direction `p - q`, and equal four-way probability produced perplexity four.
- **What it does not prove:** It does not establish that a trained language model is accurate, calibrated, useful, safe, or better than another model. It also does not verify PyTorch autograd, causal target alignment, masking, or learning over a dataset; later sessions test those claims.

Notice the discipline: observations name executed results, interpretations explain mechanisms, and broad model-quality claims remain outside the evidence.
</details>

## Final synthesis question

Softmax ignores a shared shift in all logits, yet cross-entropy can still strongly punish a confident error. How can both facts be true at the same time? Frame your answer in terms of **relative gaps**, **target probability**, and **gradient direction**.

<details>
<summary><strong>Reference synthesis</strong></summary>

A shared shift changes no relative gap, so it changes neither probabilities nor cross-entropy. A confident error is different: it creates a large gap favoring the wrong token over the target, driving the target probability toward zero and `-log(p_target)` upward.

The gradient then reflects the same relative mistake. The target coordinate approaches `-1`, so gradient descent strongly raises the target logit; probability mass concentrated on wrong candidates gives them positive gradients, so gradient descent lowers them. Shift invariance and strong error correction are therefore compatible: the loss ignores directions that change no competition and reacts strongly to directions that distort the competition against the observed target.
</details>